# inferah-bench v0.3

Causal correctness of LLM answers to *"why did this metric change?"* — bare
SQL agent (A) vs agent + exhaustive docs (B) vs agent + deterministic engine
(D / D2 / D3). Ground truth by construction; deterministic scoring; no
LLM-judge.

| arm | what it is |
|---|---|
| **A** | bare agent, `run_sql` only |
| **B** | agent + exhaustive semantic-layer docs, `run_sql` |
| **D** | agent + engine, LLM narrates a TEXT report (v0) |
| **D2** | agent + engine, LLM narrates JSON + completeness gate (v0.2) |
| **D3** | engine + **code** narrator — zero LLM in the answer loop (v0.3) |

Arms A/B run per model in the grid; engine arms D/D2/D3 carry their own tag
(D3 is deterministic → stability 1.00 by construction, $0).

## Methodology — what changed

| version | change | arms touched |
|---|---|---|
| v0 | initial grid A/B/D | A, B, D |
| v0.2 | audit (26 narrator-err, 0 engine-err) → D2: JSON interface + completeness gate | D2 only |
| v0.3 | D3 (code narrator, no LLM); multi-model A/B transport; run-it-yourself packaging | D3 only |

A/B prompts, cases, labels, scoring frozen since v0. Arm D iterated
(v0→D2→D3); all versions kept in the table.

## 1 · Setup (Postgres from `$PG_URL`, engine as installed package)

In [ ]:
import json, pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd()))
from bench.config import pg_url
from sqlalchemy import create_engine, text
PG_URL = pg_url()
eng = create_engine(PG_URL)
seeded = eng.connect().execute(text(
    "SELECT EXISTS (SELECT 1 FROM information_schema.tables "
    "WHERE table_schema='case_28' AND table_name='orders')")).scalar()
LABELS = json.loads(pathlib.Path("cases/labels.json").read_text())
print("postgres:", PG_URL, "| cases seeded:", seeded, "| labels:", len(LABELS))
if not seeded:
    from cases.generators import seed_postgres
    seed_postgres(eng)

## 2 · Score the cache (no API calls)

The full grid is cached in `results/raw.jsonl`. This re-scores it; to add a
model, set `BENCH_MODELS` and run `make full-run` first.

In [ ]:
from bench.runner import load_cache, fact_banks, score_records, usage_summary
records = list(load_cache().values())
banks = fact_banks(LABELS, PG_URL)
scores = score_records(records, LABELS, banks)
scores.to_parquet("results/scores.parquet")
scores["col"] = scores.apply(
    lambda r: r.arm if r.arm.startswith("D") else f"{r.arm}/{r.model}", axis=1)
print(len(scores), "scored records across:", sorted(scores.col.unique()))

## 3 · Main table — score by failure type

In [ ]:
import pandas as pd
pd.set_option("display.precision", 2)
order = [c for c in ["A/claude-sonnet-4-6", "A/gpt-5.2", "A/gpt-5-mini",
                     "B/claude-sonnet-4-6", "B/gpt-5.2", "B/gpt-5-mini",
                     "D", "D2", "D3"] if c in scores.col.unique()]
order += [c for c in sorted(scores.col.unique()) if c not in order]
main = scores.pivot_table(index="type", columns="col", values="score",
                          aggfunc="mean")[order]
main.loc["ALL"] = scores.groupby("col")["score"].mean()
display(main.style.background_gradient(axis=None, cmap="RdYlGn", vmin=0, vmax=1)
        .format("{:.2f}"))

In [ ]:
comp = scores.groupby("col")[["action_correct", "driver_score",
                              "grounding_pass", "sum_share_pass"]].mean()
comp["stability"] = scores.groupby("col")["modal_share"].mean()
comp["$/run"] = scores.groupby("col")["cost_usd"].mean()
display(comp.loc[order].round(3))

## 4 · Cross-model check (arms A/B)

If the picture holds across providers (docs don't lift the average, the tail
falls), the claim generalizes beyond one model. If it differs, that's its own
finding — the per-model deltas are below.

In [ ]:
ab = scores[scores.arm.isin(["A", "B"])]
models = sorted(ab.model.unique())
if len(models) > 1:
    by_model = ab.pivot_table(index=["arm", "type"], columns="model",
                              values="score", aggfunc="mean")
    display(by_model.round(2))
    allm = ab.pivot_table(index="arm", columns="model", values="score",
                          aggfunc="mean")
    print("ALL score by arm x model:")
    display(allm.round(3))
else:
    print(f"Only one model in the grid so far: {models}. "
          f"Add models via BENCH_MODELS + `make full-run`, then re-run.")

## 5 · D2 vs D3 — does the code narrator beat the LLM narrator?

In [ ]:
d = scores[scores.arm.isin(["D2", "D3"])]
piv = d.pivot_table(index="type", columns="arm", values="score", aggfunc="mean")
piv.loc["ALL"] = d.groupby("arm")["score"].mean()
piv["Δ D2→D3"] = (piv["D3"] - piv["D2"]).round(3)
display(piv.round(3))
# case-level: any case where the LLM narrator BEAT the code mapping?
d2m = scores[scores.arm == "D2"].groupby("case_id").score.mean()
d3m = scores[scores.arm == "D3"].set_index("case_id").score
regress = [(c, round(d2m[c], 2), round(d3m[c], 2)) for c in d3m.index
           if d3m[c] < d2m.get(c, 0) - 1e-9]
print("cases where D3 (code) < D2 (LLM):", regress or
      "none — code narrator is >= LLM narrator everywhere "
      "(LLM only added noise, never repaired the engine)")

## 6 · Audit of D's v0 failures (Fix 0)

In [ ]:
audit = json.loads(pathlib.Path("results/audit_d_v0.json").read_text())
from bench.audit_d import bucket_table
display(bucket_table(audit))
assert not [r for r in audit if r["bucket"] == "engine_error"]
print("engine_error: 0 — the engine never named the wrong driver; "
      "every D failure was the LLM narrator or a label stricter than greedy walk")

## 7 · Stability — D3 is 28/28 by construction

In [ ]:
stab = scores.groupby(["case_id", "col"]).modal_share.first().reset_index()
display(stab.groupby("col").modal_share.mean().reindex(order).round(3).to_frame("modal_share"))
bad = stab[(stab.col == "D3") & (stab.modal_share < 1.0)]
print("D3 non-1.0:", bad.case_id.tolist() or "none — deterministic ✓")

## 8 · Grounding failures (fabricated numbers)

In [ ]:
g = scores.pivot_table(index="type", columns="col", values="grounding_pass",
                       aggfunc="mean")[order]
g.loc["ALL"] = scores.groupby("col")["grounding_pass"].mean()
display((1 - g).style.background_gradient(axis=None, cmap="Reds", vmin=0,
                                          vmax=1).format("{:.0%}"))

## 9 · Conclusions

**(а) Exhaustive documentation does not raise the average — it reshuffles
errors.** ALL: A 0.70 → B 0.70. Docs cured the tail (T6 0.24→0.60,
T7 0.44→0.88) and broke the middle by the same caution (T2 0.89→0.41: false
"no driver" on real uniformly-spread drivers). On the SQL arms B answered
"no_driver"/"abstain" on real-driver cases far more than A — the price of
loading it with completeness rules and mix-effect warnings.

**(б) The bare agent confidently explains noise.** T6 = 0.24: in ~70% of runs
arm A produced a full driver story for a sub-significance move; grounding
failures reach 75% on T5 and 40% on T4 — it invents the supporting numbers.

**(в) The deterministic layer wins on honesty, stability, and cost.** D2/D3
action_correct = 1.00 (no false explain, no missed significance); T4 (abstain)
and T6 (no_driver) clean 1.00; one row/day-count gate took T7 0.40→0.95; $/run
~$0.01 (D2) and $0 (D3) vs ~$0.045 for the SQL arms.

**(г) Open / unresolved.** T5 compound 0.67–0.70: the engine's greedy walk
reports the single dominant driver, not both — engine roadmap (multi-driver
decomposition), not a scoring artifact. Category-mix Simpson (case_10)
decomposes as a numerically-true volume move because the pack's rate/mix axis
is order_type.

**(д) Code narrator vs LLM narrator.** Replacing the D2 LLM narrator with a
pure code mapping (D3) was a *strict* improvement everywhere — ALL 0.90→0.94,
stability 0.94→1.00, sum_share 0.96→1.00, with zero regressions at the case
level. The v0 audit already showed 0 engine_errors; D3 confirms the LLM was
only *adding noise* to a correct decomposition, never repairing it. The honest
conclusion: once a deterministic engine produces the answer, the narration
belongs in code, not in a model.

*(Multi-model A/B columns generalize (а)–(б) across providers when present;
see §4.)*